# Análise da Rede de Assistência ao Câncer de Pulmão — Ceará 2024

Este notebook implementa as Fases II a V da metodologia proposta no artigo.

## 0. Imports e configurações

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import networkx as nx
import webbrowser

sys.path.append(os.path.abspath('.'))
import graph as gf

DADOS   = '../dados'
OUTPUT  = '../output'
os.makedirs(OUTPUT, exist_ok=True)

## 1. Carregar dados

In [2]:
# Dados de internações com nomes de hospitais e municípios
df_pulmao = pd.read_csv(f'{DADOS}/dados_sih_2024_pulmao.csv', low_memory=False)

cnes_mapping  = pd.read_csv(f'{DADOS}/CNESBR.csv')
munic_mapping = pd.read_csv(f'{DADOS}/MUNICSBR.csv')
geo_data      = pd.read_csv(f'{DADOS}/geograf.csv')
df_porte      = pd.read_csv(f'{DADOS}/cnes_porte_CE_2024.csv')

# Merge para obter nomes
df_pulmao['CNES']      = df_pulmao['CNES'].astype(str)
df_pulmao['MUNIC_RES'] = df_pulmao['MUNIC_RES'].astype(str)
cnes_mapping['CNES']   = cnes_mapping['CNES'].astype(str)
munic_mapping['COD']   = munic_mapping['COD'].astype(str)

df_pulmao = df_pulmao.merge(
    cnes_mapping.rename(columns={'NOMEFANT': 'HOSPITAL'})[['CNES', 'HOSPITAL']],
    how='left', on='CNES'
)
df_pulmao = df_pulmao.merge(
    munic_mapping.rename(columns={'MUNIC': 'MUNICIPIO', 'COD': 'MUNIC_RES'})[['MUNIC_RES', 'MUNICIPIO']],
    how='left', on='MUNIC_RES'
)
df_pulmao = df_pulmao.dropna(subset=['MUNICIPIO', 'HOSPITAL'])

print(f'Total de internações: {len(df_pulmao)}')
print(df_pulmao['TIPO_PROC'].value_counts())

Total de internações: 1268
TIPO_PROC
Tratamento Clinico    1048
Cirurgia               166
Intercorrencia          54
Name: count, dtype: int64


## 2. Fase II — Criação dos grafos e estatísticas

In [3]:
# Grafo completo
G_total = gf.criar_grafo(geo_data, df_pulmao)
gf.estatisticas_grafo(G_total, titulo='Grafo Completo — 2024')
mapa = gf.plotar_grafo_folium(G_total, output_path='../output/grafo_total.html')
mapa

[AVISO] Coordenadas não encontradas: UNID MISTA NOSSA SRA DAS ANGUSTIAS
[AVISO] Coordenadas não encontradas: MANAUS, AM

  Grafo Completo — 2024
  Nós (total):      226
  Municípios:       134
  Hospitais:        92
  Arestas:          279
  Grau médio:       2.4690
  Densidade:        0.005487
  Comp. fracas:     17
  Comp. fortes:     226

Mapa salvo em: ../output/grafo_total.html


In [4]:
G_trat = gf.criar_grafo_por_tipo(geo_data, df_pulmao, 'Tratamento Clinico')
G_int  = gf.criar_grafo_por_tipo(geo_data, df_pulmao, 'Intercorrencia')
G_cir  = gf.criar_grafo_por_tipo(geo_data, df_pulmao, 'Cirurgia')

gf.estatisticas_grafo(G_trat, titulo='Tratamento Clínico')
gf.estatisticas_grafo(G_int,  titulo='Intercorrência')
gf.estatisticas_grafo(G_cir,  titulo='Cirurgia')

[AVISO] Coordenadas não encontradas: MANAUS, AM
[AVISO] Coordenadas não encontradas: UNID MISTA NOSSA SRA DAS ANGUSTIAS

  Tratamento Clínico
  Nós (total):      205
  Municípios:       120
  Hospitais:        85
  Arestas:          225
  Grau médio:       2.1951
  Densidade:        0.005380
  Comp. fracas:     23
  Comp. fortes:     205


  Intercorrência
  Nós (total):      48
  Municípios:       26
  Hospitais:        22
  Arestas:          31
  Grau médio:       1.2917
  Densidade:        0.013741
  Comp. fracas:     17
  Comp. fortes:     48


  Cirurgia
  Nós (total):      69
  Municípios:       57
  Hospitais:        12
  Arestas:          81
  Grau médio:       2.3478
  Densidade:        0.017263
  Comp. fracas:     2
  Comp. fortes:     69



Grafo Tratamento Clínico

In [5]:
mapa_trat = gf.plotar_grafo_folium(G_trat, '../output/grafo_tratamento.html')
mapa_trat

Mapa salvo em: ../output/grafo_tratamento.html


Grafo Intercorrencia

In [6]:
mapa_int = gf.plotar_grafo_folium(G_int, '../output/grafo_intercorrencia.html')
mapa_int

Mapa salvo em: ../output/grafo_intercorrencia.html


Grafo Cirurgia

In [7]:
mapa_cir = gf.plotar_grafo_folium(G_cir, '../output/grafo_cirurgia.html')
mapa_cir

Mapa salvo em: ../output/grafo_cirurgia.html


## 3. Fase II — Identificação de hospitais sobrecarregados

In [8]:
def identificar_sobrecarregados(G, df_porte, cnes_mapping, titulo=''):
    
    hospitais = [
        (n, sum(G[u][n]['weight'] for u in G.predecessors(n)))
        for n, d in G.nodes(data=True)
        if d.get('bipartite') == 1
    ]

    df = pd.DataFrame(hospitais, columns=['HOSPITAL', 'INTERNACOES'])

    mapa_hospitais = cnes_mapping.rename(
        columns={'NOMEFANT': 'HOSPITAL'}
    )[['CNES', 'HOSPITAL']]

    df = df.merge(
        mapa_hospitais,
        on='HOSPITAL',
        how='left'
    )

    df_porte['CNES'] = df_porte['CNES'].astype(str)
    df['CNES'] = df['CNES'].astype(str)

    df = df.merge(
        df_porte[['CNES', 'PORTE']],
        on='CNES',
        how='left'
    )

    df['PORTE'] = df['PORTE'].fillna('Desconhecido')

    thresholds = {}
    for porte, grupo in df.groupby('PORTE'):
        thresholds[porte] = grupo['INTERNACOES'].mean() + grupo['INTERNACOES'].std()

    df['THRESHOLD'] = df['PORTE'].map(thresholds)
    df['SOBRECARREGADO'] = df['INTERNACOES'] > df['THRESHOLD']

    print(f'\n--- {titulo} ---')
    print(
        df[df['SOBRECARREGADO']]
        [['HOSPITAL', 'INTERNACOES', 'PORTE', 'THRESHOLD']]
        .to_string(index=False)
    )

    return df

df_sobre_total = identificar_sobrecarregados(G_total, df_porte.copy(), cnes_mapping.copy(), 'Rede Geral')


--- Rede Geral ---
                                                HOSPITAL  INTERNACOES   PORTE  THRESHOLD
HM HOSPITAL DE MESSEJANA DR CARLOS ALBERTO STUDART GOMES          683  Grande 234.118777
            HOSPITAL MUNICIPAL SENADOR CARLOS JEREISSATI            4 Pequeno   3.684794
            HOSPITAL MUNICIPAL SENADOR CARLOS JEREISSATI            4 Pequeno   3.684794
                                HOSPITAL HAROLDO JUACABA           49   Medio  14.342530
                      HOSPITAL E CASA DE SAUDE DE RUSSAS           22   Medio  14.342530
                        HOSPITAL DR ALBERTO FEITOSA LIMA           16   Medio  14.342530
                                   HOSPITAL SAO RAIMUNDO           15   Medio  14.342530
                                   HOSPITAL SAO RAIMUNDO           15   Medio  14.342530
                                   HOSPITAL SAO RAIMUNDO           15   Medio  14.342530
                        SOCIEDADE HOSPITALAR PE DIONISIO            6 Pequeno   3.684794
 

## 4. Visualização — Mapas interativos

In [9]:
sob_total = df_sobre_total[df_sobre_total['SOBRECARREGADO']]['HOSPITAL'].tolist()

gf.plotar_grafo_folium(G_total, f'{OUTPUT}/grafo_total.html', sobrecarregados=sob_total)

Mapa salvo em: ../output/grafo_total.html


## 5. Métricas dos Grafos

In [11]:
def calcular_metricas(G):
    graus = [grau for _, grau in G.degree()]

    return {
        'Número de nós': G.number_of_nodes(),
        'Número de arestas': G.number_of_edges(),
        'Grau médio': round(sum(graus) / len(graus), 2),
        'Grau máximo': max(graus),
        'Densidade': nx.density(G),
        'N° de Componentes Conectados': nx.number_weakly_connected_components(G)
    }

metricas_total = calcular_metricas(G_total)
metricas_trat = calcular_metricas(G_trat)
metricas_int = calcular_metricas(G_int)
metricas_cir = calcular_metricas(G_cir)

df_metricas = pd.DataFrame({
    'Completo': metricas_total,
    'Tratamento': metricas_trat,
    'Intercorrências': metricas_int,
    'Cirurgias': metricas_cir
})

df_metricas

,Completo,Tratamento,Intercorrências,Cirurgias
Número de nós,226.000000,205.00000,48.000000,69.000000
Número de arestas,279.000000,225.00000,31.000000,81.000000
Grau médio,2.470000,2.20000,1.290000,2.350000
Grau máximo,73.000000,64.00000,7.000000,31.000000
Densidade,0.005487,0.00538,0.013741,0.017263
N° de Componentes Conectados,17.000000,23.00000,17.000000,2.000000
